In [4]:
import os
import requests
import json
import re
import base64
import csv
import pandas as pd

/Users/rachanavenatiicloud.com/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [6]:
csv_file_path = "../outputs/cleaned_b64file.csv"
outputpath="../outputs/llava_img_senti.csv"
llava_api_url = "http://gammaweb05.medien.uni-weimar.de:11439/api/generate"


##REMOVING dataurl of base64

In [ ]:
##REMOVING dataurl of base64

df = pd.read_csv("../dataset/splits/test_data_img.csv")

def clean_base64(data_url):
    if isinstance(data_url, str) and "base64," in data_url:
        return re.sub(r'^data:.*?;base64,', '', data_url)
    return data_url  

df["filename"] = df["filename"].apply(clean_base64)

df.to_csv("../outputs/cleaned_b64file.csv", index=False)


##Actual CALL to Gammaweb

In [18]:
def llm_request(prompt, base64_image):
    print("inside llm request")
    data = {
        "model": "llava:latest",
        "prompt": prompt,
        "images": [base64_image], 
        "stream": False,
        "options": {
            "num_ctx": 32768,
            "temperature":0.0,
            

        }
    }
    response = requests.post(llava_api_url, json=data)
    print(response.status_code)
    if response.status_code == 200:
        # result = response.json().get('response')
        # match = re.search(r"\d+", result)
        # if match:
        response=response.json()
        return  response["response"]# Normalize to 0-1 range
        
        
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None


In [40]:
import pandas as pd

def process_csv(csv_file_path,outputpath):
    # c=0
    results = []
    try:
        df = pd.read_csv(csv_file_path)
        print("CSV Headers:", df.columns.tolist())  
        if "llava_response" not in df.columns:
            df["llava_response"] = ""
        for index, row in df.iterrows():

            # if c==3:
            #   break
            try:
                base64_image = row['filename']
                #print(c)
                #score = process_image(base64_image)
                prompt="""Give me the sentiment of the image, i.e., which emotion is evoked upon seeing the image sent to you.

Definitions:
- **POSITIVE**: The image evokes happiness, joy, warmth, excitement, or inspiration,sarcasam,comics,animation,animals, smiling faces, natural beauty, celebrations,art,fashion,photography,wildlife, or cozy settings.
- **NEGATIVE**: The image evokes sadness, fear, discomfort, distress, or danger. It may include blood,weapons, crying faces, destruction, loneliness, or threatening situations,depression,suicidal,crime.
- **NEUTRAL**: The image does not evoke strong positive or negative emotions. It includes news-related images, technical images, or plain landscapes with balanced elements.

You should respond only in three labels: **POSITIVE, NEGATIVE, or NEUTRAL**.

DO NOT GIVE ME REASONING, JUST GIVE A SINGLE WORD RESPONSE.
Always respond in **lowercase**.
"""
                sentiment_score = llm_request(prompt, base64_image)
                df.at[index, "llava_response"] = sentiment_score
                results.append({
                    'sentiment_score': sentiment_score
                })
                # c=c+1
        
            except KeyError as e:
                print(f"Skipping row {index} due to missing key: {e}")
            except Exception as e:
                print(f"Error processing row {index}: {e}")
        df.to_csv(outputpath, index=False)
        print("Updated CSV file successfully saved!")

    except Exception as e:
        print(f"Error reading CSV file: {e}")
    return results

In [ ]:
all_results = process_csv(csv_file_path,outputpath)

for result in all_results:
    print(f"Sentiment Score: {result['sentiment_score']}")

CSV Headers: ['annotation_id', 'annotator', 'author', 'author_avatar_url', 'author_fullname', 'body', 'created_at', 'decoded_image_path', 'filename', 'hashtags', 'id', 'image_url', 'label_post', 'lead_time', 'location_city', 'location_latlong', 'location_name', 'media_url', 'num_comments', 'num_likes', 'num_media', 'parent_id', 'post_mood_PNN', 'post_mood_PNN_text', 'post_source_domain', 'thread_id', 'timestamp', 'type', 'unix_timestamp', 'updated_at', 'url']
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside llm request
200
inside 